# Unidad 32 — Programación dinámica

Semana 5 · Clase 1 · Fuente: PDF parte 1, págs. 4–38 (la pág. 39 es la portada de la Unidad 33 y no se incluye)

[Índice de la semana](../semana-05.ipynb) | [Anterior](../semana-05.ipynb) | [Siguiente](u33-backtracking.ipynb)

In [1]:
def verificar(descripcion, obtenido, esperado):
    """Compara el resultado de un ejercicio con el esperado e informa sin detener el notebook."""
    if obtenido == esperado:
        print(f"Correcto: {descripcion}")
    else:
        print(f"Incorrecto: {descripcion}. Se esperaba {esperado!r} y se obtuvo {obtenido!r}")

## Objetivos de la unidad

- Entender la programación dinámica y calcular la secuencia de Fibonacci con ella.
- Calcular el coeficiente binomial mediante el triángulo de Pascal.
- Resolver problemas de optimización usando la relación recursiva de forma ascendente (bottom-up).

Palabras clave del PDF: programación dinámica, recursivo, propiedad, enfoque, bottom-up, memorización.

**Conceptos previos (según el PDF):** definir subproblemas de forma recursiva con dividir y vencer (Unidad 31), implementar funciones recursivas a partir de relaciones de recurrencia y guardar valores en una tabla con listas y diccionarios.

**Referencia:** PDF parte 1, págs. 4–7

## Contexto

Leonardo de Pisa, conocido como Fibonacci, pensó en esta regla mientras reflexionaba sobre la población de conejos:

- En el primer mes solo existe una pareja de conejos.
- En el segundo mes, esa pareja da a luz a una pareja de crías.
- En el tercer mes, dos parejas dan a luz a dos parejas de crías (según el PDF, que simplifica el crecimiento).
- Se supone que una pareja reproducible da a luz a una pareja bebé cada mes y que los conejos no mueren.

La pregunta es: ¿cuántas parejas totales de conejos habrá en el mes N? La respuesta es el N-ésimo número de Fibonacci, un problema muy significativo en la teoría matemática. Se define con dos valores iniciales y una recurrencia:

- F(0) = 0
- F(1) = 1
- F(n) = F(n-1) + F(n-2)

La secuencia es: 0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610, ...

**Nota sobre el PDF:** la diapositiva de la pág. 10 escribe la recurrencia como `F(n-1)+F(n-1)`. Es un error de traducción o de edición: lo correcto es `F(n-1)+F(n-2)`, como confirma el código `fib1` de la pág. 11.

**Referencia:** PDF parte 1, págs. 8–10

## Fibonacci recursivo: `fib1` y los subproblemas superpuestos

El problema se resuelve de forma directa con la ecuación recursiva. La función `fib1(n)` del PDF implementa la definición tal cual:

- Si `n` es 0 o 1, la condición de terminación devuelve `n`.
- En otro caso, devuelve la suma de los términos `n-1` y `n-2`.

In [2]:
def fib1(n):
    if n == 0 or n == 1:
        return n
    else:
        return fib1(n - 1) + fib1(n - 2)

# El PDF lee N con input() (ejemplo: 10 -> 55); aquí usamos valores fijos
print(fib1(10))
print([fib1(i) for i in range(16)])

55
[0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610]


Este método tiene una grave ineficacia. Al llamar a `fib1(5)` el árbol de llamadas es:

```
fib1(5)
├── fib1(4)
│   ├── fib1(3)
│   │   ├── fib1(2)
│   │   │   ├── fib1(1)
│   │   │   └── fib1(0)
│   │   └── fib1(1)
│   └── fib1(2)
│       ├── fib1(1)
│       └── fib1(0)
└── fib1(3)
    ├── fib1(2)
    │   ├── fib1(1)
    │   └── fib1(0)
    └── fib1(1)
```

`fib1(3)` se calcula dos veces, `fib1(2)` tres veces y `fib1(1)` cinco veces. Este problema de llamadas recursivas duplicadas se denomina **subproblemas superpuestos** (overlapping subproblems). El PDF dibuja solo parte del árbol (págs. 12–13) para resaltar los repetidos.

**Ampliación (no está en el PDF):** podemos medir la duplicación contando llamadas. El número de llamadas crece casi como la propia secuencia (exponencialmente).

In [3]:
# Ampliación: contar cuántas veces se llama a cada subproblema en fib1(5)
from collections import Counter

llamadas = Counter()

def fib1_contado(n):
    llamadas[n] += 1
    if n == 0 or n == 1:
        return n
    return fib1_contado(n - 1) + fib1_contado(n - 2)

fib1_contado(5)
for n in sorted(llamadas, reverse=True):
    print(f"fib1({n}) se llamó {llamadas[n]} vez/veces")
print("Total de llamadas:", sum(llamadas.values()))

# El total crece rápido con n
for n in (10, 15, 20, 25):
    llamadas.clear()
    fib1_contado(n)
    print(f"n = {n:2d} -> {sum(llamadas.values()):>7,} llamadas")

fib1(5) se llamó 1 vez/veces
fib1(4) se llamó 1 vez/veces
fib1(3) se llamó 2 vez/veces
fib1(2) se llamó 3 vez/veces
fib1(1) se llamó 5 vez/veces
fib1(0) se llamó 3 vez/veces
Total de llamadas: 15
n = 10 ->     177 llamadas
n = 15 ->   1,973 llamadas
n = 20 ->  21,891 llamadas
n = 25 -> 242,785 llamadas


**Clave:** la causa de la lentitud no es la recursión en sí, sino que el mismo subproblema se resuelve muchas veces.

**Error común:** creer que una función recursiva siempre es lenta. `fib1` lo es por los subproblemas superpuestos; una recursión sin repetición (por ejemplo, factorial) es lineal.

**Referencia:** PDF parte 1, págs. 11–13 y 23

## Programación dinámica frente a dividir y vencer

La programación dinámica es parecida a dividir y vencer (Unidad 31) en que ambas plantean una solución recursiva al problema. La diferencia es que dividir y vencer **repite cálculos** cuando los subproblemas se solapan, mientras que la programación dinámica **almacena** los subproblemas en una variable o tabla para no recalcular los ya resueltos.

| Enfoque | Dirección | Qué hace con los subproblemas repetidos |
|---|---|---|
| Dividir y vencer | Top-down (de arriba hacia abajo) | Los recalcula cada vez |
| Programación dinámica | Bottom-up (de abajo hacia arriba) | Los guarda en una tabla y los reutiliza |

### Cómo se diseña un algoritmo de programación dinámica

La programación dinámica es una técnica de diseño que analiza las características recursivas del problema y obtiene la respuesta de forma ascendente. El procedimiento del PDF:

1. Establecer una ecuación recursiva que calcule la respuesta al caso de entrada.
2. Obtener la respuesta al caso completo resolviendo primero los casos pequeños (ascendente).

La técnica de guardar las respuestas de los subproblemas superpuestos en una tabla (o memoria) se denomina **memorización** (memoization). Cuando se vuelve a encontrar el mismo subproblema, se toma directamente de la tabla sin recalcular.

**Ampliación (no está en el PDF):** para que la programación dinámica sea aplicable el problema debe tener dos propiedades: *subestructura óptima* (la solución óptima se construye con soluciones óptimas de subproblemas) y *subproblemas superpuestos* (los mismos subproblemas aparecen una y otra vez). Estrictamente, el término "memorización" se usa para la versión recursiva con caché (top-down con tabla), y "tabulación" para el llenado por bucles (bottom-up). El PDF llama "memorización" a ambas ideas.

**Clave:** programación dinámica = recurrencia + tabla que evita recalcular.

**Para el quiz:** dividir y vencer es top-down y recalcula; programación dinámica es bottom-up y memoriza.

**Referencia:** PDF parte 1, págs. 14–17

## Fibonacci con memorización: `fib2` y `fib3`

### `fib2`: memorización con un diccionario

`F` es un diccionario inicializado con `F[0] = 0` y `F[1] = 1`.

- Si `n` ya está en `F`, es un subproblema ya calculado y se devuelve `F[n]` de inmediato.
- Si no está, se hace la llamada recursiva, se guarda el resultado en `F[n]` y se devuelve.

In [4]:
F = {0: 0, 1: 1}

def fib2(n):
    if n in F:
        return F[n]
    else:
        F[n] = fib2(n - 1) + fib2(n - 2)
        return F[n]

print(fib2(10))
print("Tabla guardada:", F)

55
Tabla guardada: {0: 0, 1: 1, 2: 1, 3: 2, 4: 3, 5: 5, 6: 8, 7: 13, 8: 21, 9: 34, 10: 55}


### `fib3`: memorización con una lista

`fib3(F, n)` usa una lista en lugar de un diccionario. Los términos aún no calculados se inicializan con `-1`, que permite detectar si un cálculo ya se hizo.

- Si `n <= 1`, devuelve `F[n]`.
- Si `F[n] < 0`, el término no se ha calculado: se hace la llamada recursiva y se guarda. Después se devuelve `F[n]`.

Al llamar a `fib3` hay que pasar la lista `F`: los elementos 0 y 1 valen 0 y 1, y los `N - 1` restantes se inicializan en `-1`.

In [5]:
def fib3(F, n):
    if n <= 1:
        return F[n]
    else:
        if F[n] < 0:
            F[n] = fib3(F, n - 1) + fib3(F, n - 2)
        return F[n]

N = 10
F = [0, 1] + [-1] * (N - 1)   # lista de largo N + 1: [0, 1, -1, -1, ...]
print(len(F), F)
print(fib3(F, N))
print(F)

11 [0, 1, -1, -1, -1, -1, -1, -1, -1, -1, -1]
55
[0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55]


**Nota sobre el PDF:** la lista `[0, 1] + [-1] * (N - 1)` tiene `N + 1` elementos (índices 0 a N), justo lo que necesita `fib3`. Los índices 0 y 1 se inicializan por separado, por eso el resto es `N - 1`.

**Error común:** en `fib2` la tabla `F` es una variable global compartida entre llamadas. Esto es una ventaja (el trabajo se reutiliza entre llamadas distintas), pero también significa que no se reinicia sola: si quieres medir tiempos justos hay que restablecerla. En `fib3`, en cambio, la lista se crea fuera y se pasa como parámetro, y debe tener al menos `n + 1` posiciones.

**Para el quiz:** `fib2` memoriza en un diccionario con valores iniciales `{0: 0, 1: 1}`; `fib3` memoriza en una lista con `-1` como marca de "no calculado" y recibe `F` como parámetro.

**Referencia:** PDF parte 1, págs. 14 y 24–27

### Comparación de rendimiento (Prueba sorpresa Q1)

La pág. 33 pide analizar el resultado de ejecutar un código y compararlo con el rendimiento de `fib1()`, `fib2()` y `fib3()`. El código de la diapositiva es la función `fib4`, que calcula Fibonacci con un bucle y solo dos variables:

```python
def fib4(n):
    if n <= 1:
        return n
    else:
        a, b = 0, 1
        for i in range(2, n + 1):
            a, b = b, a + b
        return b
```

Esta es la versión ascendente pura (bottom-up): parte de `F(0)` y `F(1)` y avanza hasta `F(n)` sin recursión ni tabla completa, porque cada término solo depende de los dos anteriores. A continuación se miden las cuatro. Para mantener la ejecución rápida se usa `n = 27` para `fib1` y valores mayores para las demás.

In [6]:
import time

def fib4(n):
    if n <= 1:
        return n
    else:
        a, b = 0, 1
        for i in range(2, n + 1):
            a, b = b, a + b
        return b

def medir(funcion, *args):
    """Devuelve (resultado, segundos) de ejecutar funcion(*args)."""
    inicio = time.perf_counter()
    resultado = funcion(*args)
    return resultado, time.perf_counter() - inicio

N = 27
resultados = []

r, t = medir(fib1, N)
resultados.append(("fib1 (recursiva)", r, t))

G = {0: 0, 1: 1}
def fib2_medida(n):
    if n in G:
        return G[n]
    G[n] = fib2_medida(n - 1) + fib2_medida(n - 2)
    return G[n]
r, t = medir(fib2_medida, N)
resultados.append(("fib2 (diccionario)", r, t))

lista = [0, 1] + [-1] * (N - 1)
r, t = medir(fib3, lista, N)
resultados.append(("fib3 (lista)", r, t))

r, t = medir(fib4, N)
resultados.append(("fib4 (bucle)", r, t))

print(f"Fibonacci de n = {N}")
for nombre, valor, seg in resultados:
    print(f"{nombre:<20} resultado = {valor}   tiempo = {seg * 1000:9.4f} ms")

Fibonacci de n = 27
fib1 (recursiva)     resultado = 196418   tiempo =   15.1515 ms
fib2 (diccionario)   resultado = 196418   tiempo =    0.0047 ms
fib3 (lista)         resultado = 196418   tiempo =    0.0027 ms
fib4 (bucle)         resultado = 196418   tiempo =    0.0013 ms


**Lectura del resultado:** los cuatro devuelven el mismo valor, pero `fib1` tarda miles de veces más que los demás porque hace cientos de miles de llamadas repetidas, mientras que `fib2`, `fib3` y `fib4` calculan cada término una sola vez. Los tiempos exactos dependen de la máquina; lo importante es el orden de magnitud.

**Ampliación (no está en el PDF):** en complejidad, `fib1` es exponencial (aproximadamente O(1,618^n)), mientras que `fib2`, `fib3` y `fib4` son O(n). `fib4` además usa memoria O(1), frente a O(n) de `fib2`/`fib3`. Las versiones recursivas con memorización tienen otro límite: la profundidad de recursión de Python (por defecto 1000).

In [7]:
# Ampliación: límite de profundidad de la recursión con memorización
import sys
print("Límite de recursión:", sys.getrecursionlimit())

H = {0: 0, 1: 1}
def fib2_h(n):
    if n in H:
        return H[n]
    H[n] = fib2_h(n - 1) + fib2_h(n - 2)
    return H[n]

try:
    fib2_h(5000)
except RecursionError as error:
    print("RecursionError con n = 5000:", error)

# La versión por bucle no tiene ese problema
print("fib4(5000) tiene", len(str(fib4(5000))), "dígitos")

Límite de recursión: 1000
RecursionError con n = 5000: maximum recursion depth exceeded
fib4(5000) tiene 1045 dígitos


**Para el quiz:** `fib1` es la única de las cuatro que recalcula subproblemas; `fib2` y `fib3` memorizan; `fib4` es ascendente con variables.

**Referencia:** PDF parte 1, págs. 32–33

## Triángulo de Pascal y coeficiente binomial

El triángulo de Pascal es un famoso algoritmo que aplica programación dinámica. Es una matriz triangular de coeficientes binomiales:

```
1
1 1
1 2 1
1 3 3 1
1 4 6 4 1
```

El coeficiente binomial C(n, k) cumple estas propiedades recursivas:

- C(n, k) = C(n-1, k-1) + C(n-1, k), si 0 < k < n
- C(n, k) = 1, si k = 0 o k = n

Cada valor es la suma de los dos que tiene encima. Con una solución recursiva (dividir y vencer), igual que en Fibonacci, aparecen subproblemas superpuestos: por ejemplo, `bino(4, 2)` llama a `bino(3, 1)` y a `bino(3, 2)`, y ambos vuelven a llamar a `bino(2, 1)`.

Para aplicar programación dinámica se usa una tabla `B[i][j]` (con `i` = n y `j` = k), donde `B[i][j] = B[i-1][j-1] + B[i-1][j]`. Los bordes (`j = 0` y `j = i`) valen 1.

```
 i\j  0  1  2  3  4
  0   1
  1   1  1
  2   1  2  1
  3   1  3  3  1
  4   1  4  6  4  1
```

**Referencia:** PDF parte 1, págs. 18–21

### `bin1`: coeficiente binomial por recursión

- Si `k` es 0 o `n`, devuelve 1.
- En otro caso, aplica la propiedad recursiva del coeficiente binomial.

In [8]:
def bin1(n, k):
    if k == 0 or n == k:
        return 1
    else:
        return bin1(n - 1, k - 1) + bin1(n - 1, k)

# El PDF lee n y k con input(); ejemplo: n = 5, k = 3
n = 5
k = 3
print(f"binomial({n}, {k}) is {bin1(n, k)}")

binomial(5, 3) is 10


**Nota sobre el PDF:** el PDF no dice qué ocurre si `k > n` o `k < 0`: `bin1` nunca llega al caso base y recursa hasta fallar. Es una entrada inválida; basta validar antes de llamar.

### `bin2`: coeficiente binomial por programación dinámica

Para los parámetros `n` y `k` se construye el triángulo en forma de lista:

- La lista `B` tiene `n + 1` filas y la fila `i` es una lista de longitud `i + 1`, con todos los valores en 0.
- Se recorre desde la fila 0 hasta la `n` (de arriba hacia abajo en el triángulo, es decir, de los casos pequeños a los grandes). Cada `B[i][j]` vale 1 en los bordes; en el interior se calcula sumando `B[i-1][j-1]` y `B[i-1][j]`, que ya estaban calculados.

In [9]:
def bin2(n, k):
    B = [[0] * (i + 1) for i in range(n + 1)]
    for i in range(n + 1):
        for j in range(i + 1):
            if j == 0 or j == i:
                B[i][j] = 1
            else:
                B[i][j] = B[i - 1][j - 1] + B[i - 1][j]
    return B[n][k]

n = 5
k = 3
print(f"binomial({n}, {k}) is {bin2(n, k)}")

# Ampliación: mostrar el triángulo completo que construye bin2
def triangulo(n):
    B = [[0] * (i + 1) for i in range(n + 1)]
    for i in range(n + 1):
        for j in range(i + 1):
            B[i][j] = 1 if j == 0 or j == i else B[i - 1][j - 1] + B[i - 1][j]
    return B

for fila in triangulo(6):
    print(" ".join(f"{v:>2}" for v in fila).center(24))

binomial(5, 3) is 10
            1           
          1  1          
         1  2  1        
       1  3  3  1       
      1  4  6  4  1     
    1  5 10 10  5  1    
   1  6 15 20 15  6  1  


**Clave:** `bin1` recalcula; `bin2` llena la tabla por filas y cada celda se obtiene con una sola suma.

**Error común:** recorrer el triángulo en un orden en que `B[i-1][...]` todavía no está calculada. Hay que ir fila por fila, de menor a mayor `i`.

**Para el quiz:** `bin2` construye `n + 1` filas, la fila `i` con `i + 1` elementos; devuelve `B[n][k]`.

**Referencia:** PDF parte 1, págs. 28–31

### Comparación de rendimiento (Prueba sorpresa Q2)

La pág. 34 pide analizar el resultado del código y compararlo con `bin1()` y `bin2()`. El código es `bin3`, que usa **una sola lista** de largo `n + 1` en lugar de una matriz y la actualiza de derecha a izquierda:

```python
def bin3(n, k):
    B = [0] * (n + 1)
    for i in range(n + 1):
        for j in range(n, -1, -1):
            if j == 0 or j == i:
                B[j] = 1
            else:
                B[j] = B[j] + B[j - 1]
    return B[k]
```

La lista `B` representa la fila actual del triángulo. Al recorrer `j` de `n` a 0, `B[j-1]` todavía conserva el valor de la fila anterior cuando se usa, igual que `B[i-1][j-1]` en `bin2`. Así se ahorra memoria: O(n) en vez de O(n^2). La diapositiva imprime el triángulo con `bin3(i, j)` para `i` de 0 a 9.

In [10]:
def bin3(n, k):
    B = [0] * (n + 1)
    for i in range(n + 1):
        for j in range(n, -1, -1):
            if j == 0 or j == i:
                B[j] = 1
            else:
                B[j] = B[j] + B[j - 1]
    return B[k]

for i in range(10):
    for j in range(i + 1):
        print(bin3(i, j), end=" ")
    print()

1 
1 1 
1 2 1 
1 3 3 1 
1 4 6 4 1 
1 5 10 10 5 1 
1 6 15 20 15 6 1 
1 7 21 35 35 21 7 1 
1 8 28 56 70 56 28 8 1 
1 9 36 84 126 126 84 36 9 1 


In [11]:
# Comparación de rendimiento y de número de llamadas (n pequeño para que sea rápido)
llamadas_bin1 = 0

def bin1_contado(n, k):
    global llamadas_bin1
    llamadas_bin1 += 1
    if k == 0 or n == k:
        return 1
    return bin1_contado(n - 1, k - 1) + bin1_contado(n - 1, k)

n, k = 22, 11
r1, t1 = medir(bin1, n, k)
r2, t2 = medir(bin2, n, k)
r3, t3 = medir(bin3, n, k)
bin1_contado(n, k)

print(f"binomial({n}, {k})")
print(f"bin1: {r1}  {t1 * 1000:9.4f} ms  ({llamadas_bin1:,} llamadas)")
print(f"bin2: {r2}  {t2 * 1000:9.4f} ms  ({(n + 1) * (n + 2) // 2} celdas calculadas)")
print(f"bin3: {r3}  {t3 * 1000:9.4f} ms")
print("Los tres coinciden:", r1 == r2 == r3)

binomial(22, 11)
bin1: 705432    34.9355 ms  (1,410,863 llamadas)
bin2: 705432     0.0146 ms  (276 celdas calculadas)
bin3: 705432     0.0130 ms
Los tres coinciden: True


**Lectura del resultado:** los tres devuelven el mismo número. `bin1` hace cientos de miles de llamadas porque repite subproblemas; `bin2` calcula una vez cada celda del triángulo y `bin3` hace lo mismo con menos memoria.

**Nota sobre el PDF:** la diapositiva de `bin3` indica `for j in range(n, -1, -1)`, que recorre `n, n-1, ..., 0`. Con esa dirección el resultado es correcto, como se comprueba arriba. Si se recorriera de izquierda a derecha, `B[j-1]` ya habría sido sobrescrito con la fila actual y el resultado sería incorrecto.

**Referencia:** PDF parte 1, pág. 34

## Optimización ascendente con una recurrencia: subarreglo de suma máxima

El tercer objetivo de la unidad es resolver un problema de optimización usando la relación recursiva de forma ascendente. El PDF lo plantea en la práctica en pareja (pág. 38):

> Dados n enteros en una matriz (lista) unidimensional, escribe un algoritmo que encuentre cuándo se maximiza la suma de los valores sucesivos (consecutivos) de la lista.

Ejemplo del PDF: `S = [-2, 1, -3, 4, -1, 2, 1, -5, 4]` y `maximum_subarray(S)` devuelve `6` (el tramo `4, -1, 2, 1`).

**Ampliación (no está en el PDF):** la recurrencia es la siguiente. Sea `mejor_hasta[i]` la mayor suma de un tramo que **termina** en la posición `i`. Entonces `mejor_hasta[i] = max(S[i], mejor_hasta[i-1] + S[i])`, porque el tramo o bien empieza en `i`, o bien extiende el mejor tramo anterior. La respuesta es el máximo de todos los `mejor_hasta[i]`. Se calcula de izquierda a derecha (bottom-up) y cada valor depende solo del anterior, así que basta una variable. Esta es la solución clásica de Kadane.

In [12]:
def maximum_subarray(S):
    # mejor_hasta: mejor suma de un tramo que termina en la posición actual
    mejor_hasta = S[0]
    mejor = S[0]
    for i in range(1, len(S)):
        mejor_hasta = max(S[i], mejor_hasta + S[i])
        mejor = max(mejor, mejor_hasta)
    return mejor

S = [-2, 1, -3, 4, -1, 2, 1, -5, 4]
M = maximum_subarray(S)
print(M)

# Ampliación: tabla de la recurrencia paso a paso
tabla = [S[0]]
for i in range(1, len(S)):
    tabla.append(max(S[i], tabla[i - 1] + S[i]))
print("S           :", S)
print("mejor_hasta :", tabla)
print("máximo      :", max(tabla))

6
S           : [-2, 1, -3, 4, -1, 2, 1, -5, 4]
mejor_hasta : [-2, 1, -2, 4, 3, 5, 6, 1, 5]
máximo      : 6


**Clave:** la recurrencia convierte un problema de "probar todos los tramos" (O(n^2) o más) en un único recorrido (O(n)).

**Error común:** inicializar el máximo en 0. Si todos los números son negativos la respuesta correcta es el mayor de ellos, no 0.

**Referencia:** PDF parte 1, pág. 38 (el enunciado y el ejemplo); la recurrencia es ampliación

**Nota sobre el PDF:** las págs. 35–37 de la sección "Programación en pareja" contienen texto genérico (en inglés y español mezclados) sobre cómo organizar la programación en parejas: turnarse en el mouse, emparejar estudiantes con habilidades similares, motivar con incentivos, evitar trampas y entornos de aprendizaje colaborativo. Resumen útil para el estudio: en la programación en pareja se trabaja **juntos** en cada parte del problema y no se "divide y vence" repartiendo el trabajo. No incluye código.

Varias diapositivas del PDF muestran el texto con caracteres dañados cuando se renderizan como imagen (títulos y párrafos ilegibles), aunque el texto se puede extraer. El código de las diapositivas sí se ve completo y se verificó visualmente.

## Proyecto de la unidad

**Problema:** ¿cuántas parejas de conejos hay en el mes N? (el problema de Fibonacci de la misión).

**Plan de programación:**

1. Definir la recurrencia: `F(0) = 0`, `F(1) = 1`, `F(n) = F(n-1) + F(n-2)`.
2. Implementar la solución recursiva (`fib1`) y observar su ineficiencia por los subproblemas superpuestos.
3. Resolver el solapamiento almacenando resultados (`fib2`, `fib3`) o avanzando de forma ascendente (`fib4`).
4. Aplicar la misma idea al triángulo de Pascal (`bin1` frente a `bin2`).
5. Comparar rendimiento con una tabla de tiempos.

El código final encapsula el `input()` del PDF en una función y ejecuta una versión no interactiva.

In [13]:
def parejas_de_conejos(mes):
    """Parejas de conejos en el mes dado, calculadas de forma ascendente."""
    if mes <= 1:
        return mes
    anterior, actual = 0, 1
    for _ in range(2, mes + 1):
        anterior, actual = actual, anterior + actual
    return actual

def programa_fibonacci(entradas):
    """Versión no interactiva: procesa una lista de meses y compara los métodos."""
    for N in entradas:
        F2 = {0: 0, 1: 1}
        def fib2_local(n):
            if n in F2:
                return F2[n]
            F2[n] = fib2_local(n - 1) + fib2_local(n - 2)
            return F2[n]
        F3 = [0, 1] + [-1] * (N - 1)
        resultados = (fib1(N), fib2_local(N), fib3(F3, N), fib4(N), parejas_de_conejos(N))
        coinciden = len(set(resultados)) == 1
        print(f"Mes {N:2d}: {resultados[0]:>6} parejas  (los cinco métodos coinciden: {coinciden})")

programa_fibonacci([1, 2, 3, 10, 15, 20])

# Para usarlo de forma interactiva, quita el comentario de la siguiente línea
# programa_fibonacci([int(input("Input a number: "))])

Mes  1:      1 parejas  (los cinco métodos coinciden: True)
Mes  2:      1 parejas  (los cinco métodos coinciden: True)
Mes  3:      2 parejas  (los cinco métodos coinciden: True)
Mes 10:     55 parejas  (los cinco métodos coinciden: True)
Mes 15:    610 parejas  (los cinco métodos coinciden: True)
Mes 20:   6765 parejas  (los cinco métodos coinciden: True)


In [14]:
# Tabla de tiempos para distintos valores de n (n acotado para que sea rápido)
print(f"{'n':>4} {'fib1 (ms)':>12} {'fib2 (ms)':>12} {'fib3 (ms)':>12} {'fib4 (ms)':>12}")
for n in (10, 15, 20, 24, 27):
    _, t1 = medir(fib1, n)
    G = {0: 0, 1: 1}
    G.clear(); G.update({0: 0, 1: 1})   # reiniciar la tabla antes de medir
    _, t2 = medir(fib2_medida, n)
    _, t3 = medir(fib3, [0, 1] + [-1] * (n - 1), n)
    _, t4 = medir(fib4, n)
    print(f"{n:>4} {t1 * 1000:>12.4f} {t2 * 1000:>12.4f} {t3 * 1000:>12.4f} {t4 * 1000:>12.4f}")

   n    fib1 (ms)    fib2 (ms)    fib3 (ms)    fib4 (ms)
  10       0.0052       0.0020       0.0009       0.0005
  15       0.0399       0.0020       0.0013       0.0004
  20       0.4628       0.0026       0.0015       0.0005
  24       3.3306       0.0033       0.0019       0.0005
  27      14.2460       0.0048       0.0023       0.0020


**Resultado esperado:** los cinco métodos devuelven los mismos valores (por ejemplo, mes 10 -> 55, mes 20 -> 6765). En los tiempos, `fib1` crece de forma explosiva con `n`, mientras que `fib2`, `fib3` y `fib4` casi no cambian.

**Referencia:** PDF parte 1, págs. 8–13 y 22–34

## Práctica

### Ejercicio 1 — Contar las llamadas de fib1

Escribe `contar_llamadas_fib1(n)` que devuelva cuántas llamadas a `fib1` se hacen en total (incluida la inicial) al calcular `fib1(n)`. Pista: `llamadas(n) = 1` si `n` es 0 o 1; en otro caso `1 + llamadas(n-1) + llamadas(n-2)`.

In [15]:
def contar_llamadas_fib1(n):
    # Escribe tu solución aquí
    pass

In [16]:
verificar("n = 1", contar_llamadas_fib1(1), 1)
verificar("n = 5", contar_llamadas_fib1(5), 15)
verificar("n = 10", contar_llamadas_fib1(10), 177)

Incorrecto: n = 1. Se esperaba 1 y se obtuvo None
Incorrecto: n = 5. Se esperaba 15 y se obtuvo None
Incorrecto: n = 10. Se esperaba 177 y se obtuvo None


<details><summary>Ver solución</summary>

```python
def contar_llamadas_fib1(n):
    if n == 0 or n == 1:
        return 1
    return 1 + contar_llamadas_fib1(n - 1) + contar_llamadas_fib1(n - 2)
```

Cada llamada cuenta como una más la de sus dos hijos. Para `n = 5` son 15 llamadas, como en el árbol del PDF.
</details>

### Ejercicio 2 — Fibonacci con una tabla (bottom-up)

Escribe `fib_tabla(n)` que calcule el n-ésimo Fibonacci llenando una lista `T` de largo `n + 1` desde `T[0]` y `T[1]` hacia adelante, sin recursión.

In [17]:
def fib_tabla(n):
    # Escribe tu solución aquí
    pass

In [18]:
verificar("n = 0", fib_tabla(0), 0)
verificar("n = 1", fib_tabla(1), 1)
verificar("n = 10", fib_tabla(10), 55)
verificar("n = 50", fib_tabla(50), 12586269025)

Incorrecto: n = 0. Se esperaba 0 y se obtuvo None
Incorrecto: n = 1. Se esperaba 1 y se obtuvo None
Incorrecto: n = 10. Se esperaba 55 y se obtuvo None
Incorrecto: n = 50. Se esperaba 12586269025 y se obtuvo None


<details><summary>Ver solución</summary>

```python
def fib_tabla(n):
    T = [0] * (n + 1)
    if n >= 1:
        T[1] = 1
    for i in range(2, n + 1):
        T[i] = T[i - 1] + T[i - 2]
    return T[n]
```

Se llena la tabla de izquierda a derecha: cada celda depende de las dos anteriores, que ya están calculadas. Se cuida el caso `n = 0`, donde `T` tiene un solo elemento.
</details>

### Ejercicio 3 — Memorización con un diccionario local

Escribe `fib_memo(n, memo=None)` que use un diccionario para memorizar, sin variables globales. Debe crear el diccionario en la primera llamada y pasarlo en las llamadas recursivas.

In [19]:
def fib_memo(n, memo=None):
    # Escribe tu solución aquí
    pass

In [20]:
verificar("n = 0", fib_memo(0), 0)
verificar("n = 12", fib_memo(12), 144)
verificar("n = 80", fib_memo(80), 23416728348467685)

Incorrecto: n = 0. Se esperaba 0 y se obtuvo None
Incorrecto: n = 12. Se esperaba 144 y se obtuvo None
Incorrecto: n = 80. Se esperaba 23416728348467685 y se obtuvo None


<details><summary>Ver solución</summary>

```python
def fib_memo(n, memo=None):
    if memo is None:
        memo = {0: 0, 1: 1}
    if n in memo:
        return memo[n]
    memo[n] = fib_memo(n - 1, memo) + fib_memo(n - 2, memo)
    return memo[n]
```

Se evita el valor por defecto mutable `{}` (que se compartiría entre llamadas) usando `None` y creando el diccionario dentro. Cada subproblema se resuelve una sola vez.
</details>

### Ejercicio 4 — Una fila del triángulo de Pascal

Escribe `fila_pascal(n)` que devuelva la fila `n` del triángulo de Pascal como lista, construyéndola fila por fila a partir de la anterior.

In [21]:
def fila_pascal(n):
    # Escribe tu solución aquí
    pass

In [22]:
verificar("fila 0", fila_pascal(0), [1])
verificar("fila 4", fila_pascal(4), [1, 4, 6, 4, 1])
verificar("fila 6", fila_pascal(6), [1, 6, 15, 20, 15, 6, 1])

Incorrecto: fila 0. Se esperaba [1] y se obtuvo None
Incorrecto: fila 4. Se esperaba [1, 4, 6, 4, 1] y se obtuvo None
Incorrecto: fila 6. Se esperaba [1, 6, 15, 20, 15, 6, 1] y se obtuvo None


<details><summary>Ver solución</summary>

```python
def fila_pascal(n):
    fila = [1]
    for i in range(1, n + 1):
        nueva = [1] * (i + 1)
        for j in range(1, i):
            nueva[j] = fila[j - 1] + fila[j]
        fila = nueva
    return fila
```

Cada fila nueva se obtiene sumando pares de la anterior; los extremos valen 1. Solo se conserva la fila anterior.
</details>

### Ejercicio 5 — Coeficiente binomial con una tabla (como bin2)

Escribe `binomial(n, k)` que use la tabla `B` de `n + 1` filas como `bin2`, pero que devuelva 0 cuando `k < 0` o `k > n` en lugar de fallar.

In [23]:
def binomial(n, k):
    # Escribe tu solución aquí
    pass

In [24]:
verificar("C(5, 3)", binomial(5, 3), 10)
verificar("C(10, 0)", binomial(10, 0), 1)
verificar("C(6, 2)", binomial(6, 2), 15)
verificar("k mayor que n", binomial(3, 5), 0)

Incorrecto: C(5, 3). Se esperaba 10 y se obtuvo None
Incorrecto: C(10, 0). Se esperaba 1 y se obtuvo None
Incorrecto: C(6, 2). Se esperaba 15 y se obtuvo None
Incorrecto: k mayor que n. Se esperaba 0 y se obtuvo None


<details><summary>Ver solución</summary>

```python
def binomial(n, k):
    if k < 0 or k > n:
        return 0
    B = [[0] * (i + 1) for i in range(n + 1)]
    for i in range(n + 1):
        for j in range(i + 1):
            if j == 0 or j == i:
                B[i][j] = 1
            else:
                B[i][j] = B[i - 1][j - 1] + B[i - 1][j]
    return B[n][k]
```

Se valida primero el rango de `k` y luego se llena la tabla igual que `bin2`.
</details>

### Ejercicio 6 — Formas de subir una escalera (ampliación)

**Ampliación (no está en el PDF).** Una escalera tiene `n` escalones y se sube de 1 en 1 o de 2 en 2. Escribe `formas_escalera(n)` con programación dinámica ascendente. Recurrencia: `formas(n) = formas(n-1) + formas(n-2)`, con `formas(0) = 1` y `formas(1) = 1`.

In [25]:
def formas_escalera(n):
    # Escribe tu solución aquí
    pass

In [26]:
verificar("n = 1", formas_escalera(1), 1)
verificar("n = 4", formas_escalera(4), 5)
verificar("n = 10", formas_escalera(10), 89)

Incorrecto: n = 1. Se esperaba 1 y se obtuvo None
Incorrecto: n = 4. Se esperaba 5 y se obtuvo None
Incorrecto: n = 10. Se esperaba 89 y se obtuvo None


<details><summary>Ver solución</summary>

```python
def formas_escalera(n):
    anterior, actual = 1, 1
    for _ in range(2, n + 1):
        anterior, actual = actual, anterior + actual
    return actual
```

Tiene la misma forma que Fibonacci: el último paso fue de 1 (queda `formas(n-1)`) o de 2 (queda `formas(n-2)`).
</details>

### Ejercicio 7 — Subarreglo de suma máxima (programación en pareja Q1)

Escribe `subarreglo_maximo(nums)` que devuelva la mayor suma de elementos consecutivos de la lista (el tramo debe tener al menos un elemento). Usa la recurrencia ascendente descrita en la unidad.

In [27]:
def subarreglo_maximo(nums):
    # Escribe tu solución aquí
    pass

In [28]:
verificar("ejemplo del PDF", subarreglo_maximo([-2, 1, -3, 4, -1, 2, 1, -5, 4]), 6)
verificar("todos negativos", subarreglo_maximo([-3, -1, -2]), -1)
verificar("todos positivos", subarreglo_maximo([1, 2, 3]), 6)

Incorrecto: ejemplo del PDF. Se esperaba 6 y se obtuvo None
Incorrecto: todos negativos. Se esperaba -1 y se obtuvo None
Incorrecto: todos positivos. Se esperaba 6 y se obtuvo None


<details><summary>Ver solución</summary>

```python
def subarreglo_maximo(nums):
    mejor_hasta = nums[0]
    mejor = nums[0]
    for x in nums[1:]:
        mejor_hasta = max(x, mejor_hasta + x)
        mejor = max(mejor, mejor_hasta)
    return mejor
```

`mejor_hasta` es la mejor suma de un tramo que termina en el elemento actual: o empieza ahí, o extiende el anterior. Se guarda el máximo global. Inicializar en `nums[0]` y no en 0 cubre el caso de todos negativos.
</details>

### Ejercicio 8 — Cambio de monedas con el mínimo de monedas (ampliación)

**Ampliación (no está en el PDF).** Dadas las denominaciones `monedas` y un `monto`, escribe `minimo_monedas(monedas, monto)` que devuelva el menor número de monedas que suman exactamente `monto`, o `-1` si es imposible. Recurrencia: `T[m] = 1 + min(T[m - c])` para cada moneda `c <= m`, con `T[0] = 0`.

In [29]:
def minimo_monedas(monedas, monto):
    # Escribe tu solución aquí
    pass

In [30]:
verificar("1, 5, 10 y monto 28", minimo_monedas([1, 5, 10], 28), 6)
verificar("2 y monto 3", minimo_monedas([2], 3), -1)
verificar("1, 3, 4 y monto 6", minimo_monedas([1, 3, 4], 6), 2)

Incorrecto: 1, 5, 10 y monto 28. Se esperaba 6 y se obtuvo None
Incorrecto: 2 y monto 3. Se esperaba -1 y se obtuvo None
Incorrecto: 1, 3, 4 y monto 6. Se esperaba 2 y se obtuvo None


<details><summary>Ver solución</summary>

```python
def minimo_monedas(monedas, monto):
    infinito = float('inf')
    T = [0] + [infinito] * monto
    for m in range(1, monto + 1):
        for c in monedas:
            if c <= m and T[m - c] + 1 < T[m]:
                T[m] = T[m - c] + 1
    return T[monto] if T[monto] != infinito else -1
```

`T[m]` guarda el mínimo para cada monto parcial. El caso `[1, 3, 4]` con monto 6 muestra por qué no sirve elegir siempre la moneda más grande (un enfoque voraz daría 4+1+1 = 3 monedas, y lo óptimo es 3+3 = 2).
</details>

## Apuntes para estudiar

**Clave**

- Programación dinámica = recurrencia + almacenamiento de subproblemas para no recalcularlos.
- Propiedades necesarias: subproblemas superpuestos y subestructura óptima.
- `fib1` (recursión pura) es exponencial; `fib2` (diccionario), `fib3` (lista con -1) y `fib4` (bucle) son lineales.
- Top-down con memorización: recursión + tabla. Bottom-up: bucles desde los casos pequeños.
- Triángulo de Pascal: `B[i][j] = B[i-1][j-1] + B[i-1][j]`, con bordes en 1. `bin2` llena la tabla; `bin3` usa una sola lista.
- Optimización ascendente: definir qué guarda cada celda de la tabla y construir la respuesta final con ellas.

**Errores comunes**

- Confundir recursión con ineficiencia: lo lento es recalcular subproblemas superpuestos.
- Olvidar la tabla o no reiniciarla entre pruebas (`F` global en `fib2`).
- Tamaño incorrecto de la lista en `fib3`: debe tener `n + 1` posiciones.
- Recorrer `bin3` de izquierda a derecha (sobrescribe valores que aún se necesitan).
- Inicializar el máximo de un subarreglo en 0 con datos todos negativos.

**Para el quiz**

- Nombres exactos: `fib1`, `fib2`, `fib3`, `fib4`, `bin1`, `bin2`, `bin3`.
- `fib3(F, n)` recibe la lista `F` como parámetro y usa `-1` como "no calculado".
- `fib2` guarda en el diccionario `F = {0: 0, 1: 1}`.
- `bin1(5, 3)` y `bin2(5, 3)` devuelven 10; `fib1(10)` devuelve 55.
- Dividir y vencer es top-down y recalcula; programación dinámica es bottom-up y memoriza.
- Ejemplo de la pareja: `maximum_subarray([-2, 1, -3, 4, -1, 2, 1, -5, 4])` devuelve 6.

## Autoevaluación

**1. ¿Por qué `fib1(30)` es mucho más lento que `fib2(30)`?**

<details><summary>Ver respuesta</summary>

Porque `fib1` resuelve repetidas veces los mismos subproblemas (subproblemas superpuestos), con un número de llamadas exponencial. `fib2` guarda cada resultado en un diccionario y lo reutiliza, por lo que calcula cada término una sola vez.
</details>

**2. ¿Qué diferencia hay entre dividir y vencer y programación dinámica?**

<details><summary>Ver respuesta</summary>

Ambos descomponen el problema de forma recursiva. Dividir y vencer es top-down y recalcula subproblemas repetidos; la programación dinámica los almacena en una tabla (memorización) y los resuelve de forma ascendente, sin recalcular.
</details>

**3. En `fib3(F, n)`, ¿para qué sirve inicializar la lista con `-1`?**

<details><summary>Ver respuesta</summary>

El `-1` marca un término como "aún no calculado". Si `F[n] < 0`, se hace la llamada recursiva y se guarda el resultado; si no, se devuelve el valor guardado directamente.
</details>

**4. ¿Cómo se calcula cada celda `B[i][j]` en `bin2` y qué valen los bordes?**

<details><summary>Ver respuesta</summary>

`B[i][j] = B[i-1][j-1] + B[i-1][j]` para las celdas interiores; los bordes (`j == 0` o `j == i`) valen 1. Como se recorre por filas, los valores de la fila anterior ya están calculados.
</details>

**5. ¿Qué dos propiedades debe tener un problema para aplicar programación dinámica?**

<details><summary>Ver respuesta</summary>

Subproblemas superpuestos (los mismos subproblemas se repiten) y subestructura óptima (la solución se construye a partir de soluciones de subproblemas). La segunda es ampliación respecto al PDF, que destaca el solapamiento y la recurrencia.
</details>

---

[Índice de la semana](../semana-05.ipynb) | [Anterior](../semana-05.ipynb) | [Siguiente](u33-backtracking.ipynb)